<a href="https://colab.research.google.com/github/Sechavez16/MachineLearning-IA/blob/main/An%C3%A1lisis_datos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicción y Análisis Espacial de Centros de Datos Globales
**Autor:** Sebastian Echavez Cadena
**Objetivo:** Desarrollar un Pipeline de Machine Learning de extremo a extremo para procesar, limpiar y modelar datos geoespaciales y categóricos de centros de datos a nivel mundial.

## Fase 1: Configuración del Entorno y Carga de Datos
En esta sección instalaremos las dependencias necesarias y descargaremos el dataset directamente desde la API de Kaggle.





In [1]:
!pip install opendatasets scikit-learn pandas numpy matplotlib seaborn -q

In [6]:
"""
Fase 1: Configuración de API, Ingesta y Carga de Datos.
Utilizamos la API oficial de Kaggle para un pipeline de ingesta automatizado.
"""
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("muted")

# 1. Configuración de seguridad para la API de Kaggle
# Creamos el directorio oculto, movemos el token y le damos permisos de lectura exclusivos
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 2. Ingesta: Descarga y extracción simultánea (--unzip)
print(" Descargando dataset desde Kaggle...")
!kaggle datasets download -d rockyt0/world-data-centers --unzip

# 3. Exploración del directorio y carga a memoria
archivos_csv = [f for f in os.listdir('.') if f.endswith('.csv')]
print(f"\nArchivos CSV disponibles en el entorno: {archivos_csv}")

if archivos_csv:
    # Usualmente queremos el archivo principal. Si hay varios, seleccionamos el primero por ahora.
    archivo_objetivo = archivos_csv[0]
    df = pd.read_csv(archivo_objetivo)

    print(f"\n Dataset '{archivo_objetivo}' cargado exitosamente. Dimensiones: {df.shape}")
    display(df.head())
else:
    print(" No se encontraron archivos de datos.")

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
 Descargando dataset desde Kaggle...
Dataset URL: https://www.kaggle.com/datasets/rockyt0/world-data-centers
License(s): ODbL-1.0
100% 232k/232k [00:00<00:00, 85.8MB/s]


Archivos CSV disponibles en el entorno: ['reported_totals.csv', 'country_summary.csv', 'data_centers.csv']

 Dataset 'reported_totals.csv' cargado exitosamente. Dimensiones: (71, 8)


,country,country_iso2,country_iso3,reported_facilities,as_of,tracker,published_by,source_url
0,United States,US,USA,3960,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...
1,United Kingdom,GB,GBR,498,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...
2,Germany,DE,DEU,470,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...
3,China,CN,CHN,365,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...
4,France,FR,FRA,335,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...


## Fase 2: Análisis Exploratorio de Datos (EDA)
Antes de construir nuestro Pipeline, necesitamos entender la naturaleza de los datos. En esta fase identificaremos los tipos de variables, cuantificaremos los valores faltantes (nulos) y analizaremos la distribución de posibles variables objetivo para entender si nos enfrentamos a un problema de clases desbalanceadas.

In [13]:
"""
Analissi de estructira y valores faltantes
"""
print("_____ 1. INFORMACION GENERAL_____");
df.info();
print("_____\n 2 ANALISIS DE VALORES NULOS_____")
#Creamos una constante para bsucar valores nulos
nulos = df.isnull().sum(); #Verifica valores nulos y los sumo con el sum
#Creamos una variable del porcentaje de nulos
porcentaje_nulos = (nulos/len(df))*100;
#Quiero visulizar de una mejor forma
#Creon un DataFrame resumen
resumen_nulos = pd.DataFrame({
    'Valores Nulos' :nulos,
    'Porcentaje (%)':porcentaje_nulos
})

_____ 1. INFORMACION GENERAL_____
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   country              71 non-null     object
 1   country_iso2         71 non-null     object
 2   country_iso3         71 non-null     object
 3   reported_facilities  71 non-null     int64 
 4   as_of                71 non-null     object
 5   tracker              71 non-null     object
 6   published_by         71 non-null     object
 7   source_url           71 non-null     object
dtypes: int64(1), object(7)
memory usage: 4.6+ KB
_____
 2 ANALISIS DE VALORES NULOS_____


In [21]:
resumen_nulos.head()

,Valores Nulos,Porcentaje (%)


In [24]:
"""
Vamos hacer consultas
"""
print("_____ Mostramos los valores de la columna 'country'_____")
#De las primera 15
print(df['country'].unique()[:15])

_____ Mostramos los valores de la columna 'country'_____
['United States' 'United Kingdom' 'Germany' 'China' 'France' 'Canada'
 'India' 'Australia' 'Japan' 'Italy' 'Brazil' 'Spain' 'Netherlands'
 'Indonesia' 'Russia']


In [28]:
#Consultamos solo para china
print("Consulta solo para China")
consulta_china =df[df['country']=='China']
consulta_china.head()

Consulta solo para China


,country,country_iso2,country_iso3,reported_facilities,as_of,tracker,published_by,source_url
3,China,CN,CHN,365,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...
23,China,CN,CHN,369,2026-03,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/ranked-the-co...


In [32]:
#Buscamos los paises que tenga en su pais
print("Paisses que empiecen con Republic")
#Busco caddena de caractres en country que tenga Republic
#Case= false ignira diferencia de mayuculas y na=false ignora nulos
consulta_republic = df[df['country'].str.contains('Republic',
                                                  case=False, na=False)]
consulta_republic.head()

Paisses que empiecen con Republic


,country,country_iso2,country_iso3,reported_facilities,as_of,tracker,published_by,source_url


In [35]:
consulta_reported_facilities_mayor_4000 = df.query('reported_facilities>4000')
consulta_reported_facilities_mayor_4000.head()


,country,country_iso2,country_iso3,reported_facilities,as_of,tracker,published_by,source_url
20,United States,US,USA,4088,2026-03,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/ranked-the-co...


In [38]:
#Vimos que Usa tiene reported_facilities es de 4088
#Verificamos cuantos paieses caen en lso rangos
limites = [0,10,20,30,40,50,100,500,5000]
etiquetas = ['0-10','11-20','21-30','31-40','41-50','51-100','101-500','501-5000']
#Creo una eqtiqueta
df['rango_resported']= pd.cut(df['reported_facilities']
                              , bins=limites,
                              labels=etiquetas,
                              include_lowest=True)
df

,country,country_iso2,country_iso3,reported_facilities,as_of,tracker,published_by,source_url,rango_resported
0,United States,US,USA,3960,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...,501-5000
1,United Kingdom,GB,GBR,498,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...,101-500
2,Germany,DE,DEU,470,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...,101-500
3,China,CN,CHN,365,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...,101-500
4,France,FR,FRA,335,2026-02-18,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/mapped-countr...,101-500
...,...,...,...,...,...,...,...,...,...
66,Georgia,GE,GEO,4,2026-03,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/ranked-the-co...,0-10
67,Bosnia and Herzegovina,BA,BIH,4,2026-03,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/ranked-the-co...,0-10
68,Monaco,MC,MCO,3,2026-03,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/ranked-the-co...,0-10
69,Azerbaijan,AZ,AZE,3,2026-03,Data Center Map,Visual Capitalist,https://www.visualcapitalist.com/ranked-the-co...,0-10


Cuando un modelo de Machine Learning se entrena con datos donde una categoría domina abrumadoramente sobre las demás (como evidenciamos en tu tabla, donde la enorme mayoría de países cae en el rango "0-10"), el modelo tiende a ignorar la clase minoritaria. Simplemente predice siempre la clase mayoritaria y obtiene un falso "alta precisión".

Para resolver el desbalance, existen tres grandes familias de técnicas. Dominar estas estrategias matemáticas y algorítmicas es fundamental en cualquier maestría de IA y demuestra una arquitectura de datos sólida:

Undersampling (Submuestreo): Elimina aleatoriamente o mediante heurísticas (como Tomek Links) registros de la clase mayoritaria hasta igualar su tamaño con la minoritaria. Es útil en bases de datos masivas en la nube donde el costo de cómputo es alto, pero tiene la gran desventaja de descartar información valiosa.

Class Weighting (Pesos algorítmicos): No modifica el dataset real ni inyecta datos. Modifica directamente la función de costo del algoritmo. Si el modelo se equivoca al predecir la clase minoritaria, recibe una penalización matemática mucho mayor que si falla en la clase mayoritaria.

Oversampling y Generación Sintética (Ej. SMOTE): En lugar de simplemente duplicar registros exactos (lo cual causa sobreajuste), algoritmos como SMOTE (Synthetic Minority Over-sampling Technique) aplican métodos numéricos usando los K-Vecinos Más Cercanos (KNN) para interpolar geométricamente nuevos puntos de datos. Se crean registros "falsos" pero estadísticamente coherentes en el espacio vectorial.

In [ ]:
!pip install imbalanced-learn -q

In [44]:
from imblearn.over_sampling import SMOTE
import pandas as pd
import numpy as np

# 1. Creamos la variable objetivo binaria (¿Tiene más de 50 centros de datos?)
df['es_mayor_50'] = (df['reported_facilities'] > 50).astype(int)

# Mostramos la distribución inicial altamente desbalanceada
print("Distribución original de clases:")
print(df['es_mayor_50'].value_counts())

# 2. Preparamos las variables para SMOTE
# Usaremos 'reported_facilities' y una representación numérica de la categoría del rango para la demostración vectorizada
X = df[['reported_facilities']].copy()
# Agregamos una característica numérica auxiliar (ej. el código hash o numérico del ISO de país) para tener un espacio bidimensional
X['country_code'] = df['country_iso3'].astype('category').cat.codes
y = df['es_mayor_50']

# 3. Aplicamos SMOTE con un número de vecinos ajustado a nuestro tamaño de datos (k_neighbors=2)
smote = SMOTE(k_neighbors=2, random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# 4. Mostramos el resultado del balanceo
print("\nDistribución de clases después de SMOTE:")
print(pd.Series(y_res).value_counts())

# Creamos un DataFrame con los datos balanceados para verlos de forma clara
df_balanceado = pd.DataFrame(X_res, columns=['reported_facilities', 'country_code'])
df_balanceado['es_mayor_50'] = y_res
display(df_balanceado.head())
print(f"\nDimensiones del dataset original: {df.shape}")
print(f"Dimensiones del dataset balanceado con SMOTE: {df_balanceado.shape}")

Distribución original de clases:
es_mayor_50
1    46
0    25
Name: count, dtype: int64

Distribución de clases después de SMOTE:
es_mayor_50
1    46
0    46
Name: count, dtype: int64


,reported_facilities,country_code,es_mayor_50
0,3960,51,1
1,498,19,1
2,470,13,1
3,365,10,1
4,335,18,1



Dimensiones del dataset original: (71, 10)
Dimensiones del dataset balanceado con SMOTE: (92, 3)
